# Smoke test — Loss 1, Loss 2, and the combined objective

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**
and drives the repo's own scripts. **3 videos, 150 epochs.**

```
python -m src.dataprep.build_feature_csv --n_videos 3
python -m src.loss1.train                        # L_NCE
python -m src.loss2.train                        # L_sim
python -m src.loss2.train --joint                # L_sim + L_NCE
```

## What this is, and is not

This is a **smoke test**: does every piece run end to end, and do the losses move in the
right direction? It is **not** an experiment. 3 videos gives ~570 rows split 2 train /
1 val — a single validation video is far too noisy to conclude anything about the
premise, and the models have ~1.5 M parameters against a few hundred rows.

Read the outputs as *"the code is wired correctly"*, not *"the idea works"*.

| Run | Objective | What a healthy curve looks like |
|---|---|---|
| Loss 1 | `L_NCE` | starts near ln(batch) ≈ 4.16 and falls |
| Loss 2 | `L_sim` (MSE) | starts near 1.0 on standardised targets and falls |
| Joint | `L_sim + L_NCE` | roughly the sum of the two, falling |

## Cost

3 videos ≈ **7 GB** of downloads. `--cleanup_raw` keeps peak disk near one video.
Budget ~6 min for the build and ~2–4 min per training run. **No GPU needed.**

## 1 — Clone and install

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -3

!pip -q install -r requirements.txt
!pip -q install projectaria-tools matplotlib

import os, json, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

print("\nrepo:", os.getcwd())
print("torch", torch.__version__, "| device", "cuda" if torch.cuda.is_available() else "cpu")
!ls src

## 2 — Upload the download-links JSON

In [ ]:
from google.colab import files
up = files.upload()                    # pick your aea_download_urls.json
URLS_JSON = "/content/" + list(up.keys())[0]
os.rename(list(up.keys())[0], URLS_JSON)

# or from Drive:
# from google.colab import drive; drive.mount('/content/drive')
# URLS_JSON = "/content/drive/MyDrive/aea/aea_download_urls.json"

print(f"{len(json.load(open(URLS_JSON))['sequences'])} videos available")

## 3 — Build the dataset: 3 videos, full length

AEA sequences average ~193 s, so at 1 FPS expect **~190 rows per video, ~570 total**.

In [ ]:
N_VIDEOS = 3
SEED     = 0
OUT_CSV  = "/content/data/feature_dataset.csv"

t0 = time.time()
!python -m src.dataprep.build_feature_csv \
    --urls_json  "{URLS_JSON}" \
    --out_csv    "{OUT_CSV}" \
    --raw_dir    /content/data/raw \
    --frames_dir /content/data/frames_1fps \
    --feat_dir   /content/data/features \
    --n_videos   {N_VIDEOS} \
    --seed       {SEED} \
    --cleanup_raw

print(f"\nbuild took {(time.time()-t0)/60:.1f} min")

df = pd.read_csv(OUT_CSV)
print(f"\n{len(df)} rows from {df['sequence'].nunique()} videos")
display(df.groupby("sequence").size().rename("rows").to_frame())
display(df[["frame_similarity", "gaze_patch_token_sim"]].describe().round(3))

## 4 — Run all three objectives

150 epochs each. With ~380 training rows and batch 64 that is ~5 batches per epoch, so
each run takes a couple of minutes on CPU.

In [ ]:
EPOCHS = 150
BATCH  = 64

R1     = "/content/runs/loss1"
R2     = "/content/runs/loss2"
RJ     = "/content/runs/joint"

print("="*78 + "\n  LOSS 1  --  InfoNCE\n" + "="*78)
!python -m src.loss1.train --csv "{OUT_CSV}" --out_dir "{R1}" \
    --epochs {EPOCHS} --batch_size {BATCH} --val_frac 0.2 --seed 0 2>&1 | tail -8

print("\n" + "="*78 + "\n  LOSS 2  --  similarity regression\n" + "="*78)
!python -m src.loss2.train --csv "{OUT_CSV}" --out_dir "{R2}" \
    --epochs {EPOCHS} --batch_size {BATCH} --val_frac 0.2 --seed 0 2>&1 | tail -14

print("\n" + "="*78 + "\n  JOINT  --  L_sim + L_NCE\n" + "="*78)
!python -m src.loss2.train --csv "{OUT_CSV}" --out_dir "{RJ}" --joint \
    --epochs {EPOCHS} --batch_size {BATCH} --val_frac 0.2 --seed 0 2>&1 | tail -14

## 5 — The three loss curves

`src/loss1` and `src/loss2` write `history.json` in slightly different shapes, so the
loader below normalises them.

In [ ]:
def load_loss1(path):
    h = json.load(open(os.path.join(path, "history.json")))
    hv = h[0]["val"] is not None
    return dict(ep=[r["epoch"] for r in h],
                tr=[r["train"]["loss"] for r in h],
                va=[r["val"]["loss"] for r in h] if hv else None,
                m_tr=[r["train"]["top1"] for r in h],
                m_va=[r["val"]["top1"] for r in h] if hv else None,
                chance=float(np.mean([r["train"]["chance"] for r in h])))

def load_loss2(path):
    h = json.load(open(os.path.join(path, "history.json")))
    hv = h[0]["val"] is not None
    return dict(ep=[r["epoch"] for r in h],
                tr=[r["train_loss"] for r in h],
                va=[r["val_loss"] for r in h] if hv else None,
                m_tr=[r["train"]["r2_mean"] for r in h],
                m_va=[r["val"]["r2_mean"] for r in h] if hv else None)

L1, L2, LJ = load_loss1(R1), load_loss2(R2), load_loss2(RJ)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))

# ---- Loss 1 -------------------------------------------------------------------
ax[0].plot(L1["ep"], L1["tr"], lw=2, label="train")
if L1["va"]: ax[0].plot(L1["ep"], L1["va"], lw=2, label="val")
ax[0].axhline(np.log(BATCH), ls="--", c="k", lw=1,
              label=f"random = ln({BATCH}) = {np.log(BATCH):.2f}")
ax[0].set_title("LOSS 1 — InfoNCE", fontweight="bold")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("L_NCE"); ax[0].legend()

# ---- Loss 2 -------------------------------------------------------------------
ax[1].plot(L2["ep"], L2["tr"], lw=2, c="tab:green", label="train")
if L2["va"]: ax[1].plot(L2["ep"], L2["va"], lw=2, c="tab:olive", label="val")
ax[1].axhline(1.0, ls="--", c="k", lw=1, label="predict-the-mean = 1.0")
ax[1].set_title("LOSS 2 — similarity MSE", fontweight="bold")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("L_sim (standardised)"); ax[1].legend()

# ---- Joint --------------------------------------------------------------------
ax[2].plot(LJ["ep"], LJ["tr"], lw=2, c="tab:purple", label="train")
if LJ["va"]: ax[2].plot(LJ["ep"], LJ["va"], lw=2, c="tab:pink", label="val")
ax[2].plot(L1["ep"], np.array(L1["tr"]) + np.array(L2["tr"]), lw=1.2, ls=":", c="gray",
           label="sum of the two separate runs")
ax[2].set_title("COMBINED — L_sim + L_NCE", fontweight="bold")
ax[2].set_xlabel("epoch"); ax[2].set_ylabel("total loss"); ax[2].legend()

plt.tight_layout(); plt.show()

for nm, H in (("Loss 1 ", L1), ("Loss 2 ", L2), ("Joint  ", LJ)):
    v = f"   val {H['va'][0]:.4f} -> {H['va'][-1]:.4f}" if H["va"] else ""
    print(f"{nm}  train {H['tr'][0]:.4f} -> {H['tr'][-1]:.4f}  "
          f"({100*(1-H['tr'][-1]/H['tr'][0]):+.1f}%){v}")

## 6 — The metrics behind the curves

A falling training loss is necessary but not sufficient — these are the numbers that say
whether anything generalises.

- **Loss 1** → retrieval top-1 vs chance (`1/batch`)
- **Loss 2 / joint** → mean R², where **≤ 0 means no better than predicting the mean**

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

ax[0].plot(L1["ep"], 100*np.array(L1["m_tr"]), lw=2, label="train")
if L1["m_va"]: ax[0].plot(L1["ep"], 100*np.array(L1["m_va"]), lw=2, label="val")
ax[0].axhline(100*L1["chance"], ls="--", c="k", lw=1, label=f"chance {100*L1['chance']:.1f}%")
ax[0].set_title("Loss 1 — retrieval top-1"); ax[0].set_xlabel("epoch")
ax[0].set_ylabel("top-1 (%)"); ax[0].legend()

ax[1].plot(L2["ep"], L2["m_tr"], lw=2, c="tab:green", label="Loss 2 train")
if L2["m_va"]: ax[1].plot(L2["ep"], L2["m_va"], lw=2, c="tab:olive", label="Loss 2 val")
if LJ["m_va"]: ax[1].plot(LJ["ep"], LJ["m_va"], lw=2, c="tab:pink", ls="--", label="joint val")
ax[1].axhline(0.0, ls="--", c="k", lw=1, label="predict-the-mean (R2 = 0)")
ax[1].set_title("Loss 2 — mean R2"); ax[1].set_xlabel("epoch")
ax[1].set_ylabel("R2"); ax[1].legend()
plt.tight_layout(); plt.show()

rows = [
    dict(run="Loss 1", final_train_loss=round(L1["tr"][-1], 4),
         final_val_loss=round(L1["va"][-1], 4) if L1["va"] else None,
         metric="top-1 %", final_val_metric=round(100*L1["m_va"][-1], 2) if L1["m_va"] else None,
         reference=round(100*L1["chance"], 2)),
    dict(run="Loss 2", final_train_loss=round(L2["tr"][-1], 4),
         final_val_loss=round(L2["va"][-1], 4) if L2["va"] else None,
         metric="R2", final_val_metric=round(L2["m_va"][-1], 3) if L2["m_va"] else None,
         reference=0.0),
    dict(run="Joint", final_train_loss=round(LJ["tr"][-1], 4),
         final_val_loss=round(LJ["va"][-1], 4) if LJ["va"] else None,
         metric="R2", final_val_metric=round(LJ["m_va"][-1], 3) if LJ["m_va"] else None,
         reference=0.0),
]
display(pd.DataFrame(rows))

## 7 — Smoke-test verdict

Checks that the **code** behaves, not that the idea works.

In [ ]:
checks = [
    ("dataset built", len(df) > 100, f"{len(df)} rows, {df['sequence'].nunique()} videos"),
    ("L1 starts near ln(batch)", abs(L1["tr"][0] - np.log(BATCH)) < 0.6,
     f"{L1['tr'][0]:.3f}  vs ln({BATCH}) = {np.log(BATCH):.3f}"),
    ("L1 train loss falls", L1["tr"][-1] < 0.8*L1["tr"][0],
     f"{L1['tr'][0]:.3f} -> {L1['tr'][-1]:.3f}"),
    ("L2 train loss falls", L2["tr"][-1] < 0.8*L2["tr"][0],
     f"{L2['tr'][0]:.3f} -> {L2['tr'][-1]:.3f}"),
    ("joint train loss falls", LJ["tr"][-1] < 0.8*LJ["tr"][0],
     f"{LJ['tr'][0]:.3f} -> {LJ['tr'][-1]:.3f}"),
    ("joint > either alone (it sums two terms)", LJ["tr"][0] > max(L1["tr"][0], L2["tr"][0]),
     f"joint {LJ['tr'][0]:.3f}  vs L1 {L1['tr'][0]:.3f} + L2 {L2['tr'][0]:.3f}"),
]

print("="*74)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name:42s} {detail}")
print("="*74)

if all(c[1] for c in checks):
    print("\n  CODE IS WIRED CORRECTLY -- all three objectives run and optimise.")
else:
    print("\n  Something is off; see the failing rows above.")

print("\n  NOTE: this says nothing about whether the PREMISE holds.")
print(f"  With {df['sequence'].nunique()} videos the split is 2 train / 1 val, and the")
print("  models carry ~1.5 M parameters against a few hundred rows -- they will memorise")
print("  the training set whether or not gaze carries any signal. Expect the validation")
print("  curves to be flat or to get worse; that is the expected outcome at this scale,")
print("  not a result.")

## 8 — *(optional)* the shuffle control

Pairs each frame pair with **another row's** gaze rates. Worth running even in a smoke
test: it confirms the control path itself works, and shows how much of the training-loss
drop is pure memorisation.

Expect its **training** loss to fall just as far, and its validation R² to stay ≤ 0.

In [ ]:
RC = "/content/runs/joint_control"
!python -m src.loss2.train --csv "{OUT_CSV}" --out_dir "{RC}" --joint --shuffle_control \
    --epochs {EPOCHS} --batch_size {BATCH} --val_frac 0.2 --seed 0 2>&1 | tail -12

LC = load_loss2(RC)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot(LJ["ep"], LJ["tr"], lw=2, c="tab:purple", label="joint: real gaze")
ax[0].plot(LC["ep"], LC["tr"], lw=2, c="tab:red", ls="--", label="joint: shuffled gaze")
ax[0].set_title("training loss — real vs shuffled", fontweight="bold")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("total loss"); ax[0].legend()

if LJ["m_va"] and LC["m_va"]:
    ax[1].plot(LJ["ep"], LJ["m_va"], lw=2, c="tab:purple", label="joint: real gaze")
    ax[1].plot(LC["ep"], LC["m_va"], lw=2, c="tab:red", ls="--", label="joint: shuffled")
ax[1].axhline(0.0, ls="--", c="k", lw=1, label="predict-the-mean")
ax[1].set_title("validation R2"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("R2")
ax[1].legend()
plt.tight_layout(); plt.show()

print(f"real     train {LJ['tr'][0]:.3f} -> {LJ['tr'][-1]:.3f}   final val R2 "
      f"{LJ['m_va'][-1]:+.3f}" if LJ["m_va"] else "")
print(f"shuffled train {LC['tr'][0]:.3f} -> {LC['tr'][-1]:.3f}   final val R2 "
      f"{LC['m_va'][-1]:+.3f}" if LC["m_va"] else "")
print("\nIf the two TRAINING curves are close, the drop is memorisation, not learning.")
print("That is expected here and is the reason a smoke test cannot settle the premise.")

---

## Once the smoke test is green

The next run should be an **experiment**, not a smoke test:

```bash
python -m src.dataprep.build_feature_csv --urls_json ... --n_videos 143 --cleanup_raw
python -m src.loss2.train --csv ... --out_dir runs/loss2_full
python -m src.loss2.train --csv ... --out_dir runs/loss2_ctrl --shuffle_control
```

143 videos at full length is roughly **27,500 rows** — about 48× this smoke test, and
enough validation videos for the R² to mean something. The download is ~360 GB, so
`--cleanup_raw` is mandatory; expect several hours.

If that is too much, 30–40 videos (~7,000 rows, ~90 GB) would already give a far more
honest answer than 3.

**Read Loss 2 first when the results come in.** Its baseline — predict the training mean —
makes the outcome unambiguous in a way Loss 1's retrieval metric is not.